# Fase 1 — Representación y espacios de color

Analizamos 50 fotografías de productos individuales y 71 escenas de supermercado. MVTec D2S se añadirá al inventario visual cuando esté descargado. La máscara HSV incluida es una demostración exploratoria, no una medición de exactitud: todavía no hay máscaras manuales para las fotos propias.

## 1. Configuración y carga de imágenes
Este cuaderno busca las imágenes de `data/propio/originales` desde la raíz del proyecto o desde `notebooks/`.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidatos = [Path.cwd(), *Path.cwd().parents]
PROYECTO = next((p for p in candidatos if (p / 'data' / 'propio' / 'originales').is_dir()), None)
if PROYECTO is None:
    raise FileNotFoundError('Abre este cuaderno desde Proyecto Integrador P1 o su carpeta notebooks.')
CARPETA_PROPIA = PROYECTO / 'data' / 'propio' / 'originales'
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
rutas = sorted(p for p in CARPETA_PROPIA.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONES)
imagenes = {p.name: cv2.imread(str(p), cv2.IMREAD_COLOR) for p in rutas}
imagenes = {nombre: imagen for nombre, imagen in imagenes.items() if imagen is not None}
print(f'Imágenes propias legibles: {len(imagenes)}')

## 2. Resolución, canales y tamaño
OpenCV decodifica estas fotos como matrices BGR de 8 bits por canal. Esto describe la matriz cargada, no certifica la profundidad original del archivo.

In [ ]:
filas = []
for nombre, bgr in imagenes.items():
    alto, ancho, canales = bgr.shape
    ruta = CARPETA_PROPIA / nombre
    filas.append({
        'imagen': nombre, 'ancho_px': ancho, 'alto_px': alto,
        'canales_cargados': canales, 'bits_por_canal_cargado': bgr.dtype.itemsize * 8,
        'memoria_matriz_MB': round(bgr.nbytes / 1_000_000, 3),
        'archivo_KB': round(ruta.stat().st_size / 1024, 1),
    })
tabla = pd.DataFrame(filas)
display(tabla)

## 3. Galería de fotografías propias
El dataset incluye fotografías de productos individuales y escenas de pasillos. La galería muestra una selección breve de ambos tipos; las 121 imágenes se mantienen disponibles para el inventario y el análisis individual.

In [ ]:
muestra_galeria = [
    "miloFront.jpeg", "lechconFrontPrin.jpeg",
    "supermercado_IMG_6084.jpg", "supermercado_IMG_6103.jpg",
    "supermercado_IMG_6125.jpg", "supermercado_IMG_6150.jpg",
]
muestra_galeria = [nombre for nombre in muestra_galeria if nombre in imagenes]
n = len(muestra_galeria)
if n:
    columnas = 3
    filas_grafico = int(np.ceil(n / columnas))
    fig, ejes = plt.subplots(filas_grafico, columnas, figsize=(13, 4 * filas_grafico))
    ejes = np.array(ejes).reshape(-1)
    for ax, nombre in zip(ejes, muestra_galeria):
        bgr = imagenes[nombre]
        ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(nombre)
        ax.axis("off")
    for ax in ejes[n:]: ax.axis("off")
    plt.tight_layout()
    plt.show()


## 4. RGB, gris y binaria
La conversión a binaria depende del umbral elegido; aquí se usa 127 solo como referencia visual.

In [ ]:
nombre_elegido = 'miloFront.jpeg' if 'miloFront.jpeg' in imagenes else next(iter(imagenes))
bgr = imagenes[nombre_elegido]
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gris = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
_, binaria = cv2.threshold(gris, 127, 255, cv2.THRESH_BINARY)
fig, ejes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, titulo, cmap in zip(ejes, [rgb, gris, binaria, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)], ['RGB', 'Escala de grises', 'Binaria (umbral 127)', nombre_elegido], [None, 'gray', 'gray', None]):
    ax.imshow(img, cmap=cmap)
    ax.set_title(titulo)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Canales HSV e histogramas
OpenCV representa H en 0–179 y S/V en 0–255 para imágenes de 8 bits.

In [ ]:
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
fig, ejes = plt.subplots(1, 3, figsize=(13, 4))
for ax, canal, titulo in zip(ejes, cv2.split(hsv), ['H (tono)', 'S (saturación)', 'V (brillo)']):
    ax.imshow(canal, cmap='gray', vmin=0, vmax=255)
    ax.set_title(titulo)
    ax.axis('off')
plt.tight_layout()
plt.show()

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for valores, etiqueta, color in zip(cv2.split(rgb), ['R','G','B'], ['red','green','blue']):
    hist, bins = np.histogram(valores, bins=256, range=(0, 256))
    ejes[0].plot(bins[:-1], hist, color=color, label=etiqueta, alpha=.8)
ejes[0].set(title='Histograma RGB', xlabel='Intensidad', ylabel='Píxeles'); ejes[0].legend()
hist_gris, bins_gris = np.histogram(gris, bins=256, range=(0, 256))
ejes[1].plot(bins_gris[:-1], hist_gris, color='dimgray')
ejes[1].set(title='Histograma en gris', xlabel='Intensidad', ylabel='Píxeles')
plt.tight_layout()
plt.show()

## 6. Segmentación HSV exploratoria
La máscara siguiente busca tonos verdes como demostración para Milo. Puede incluir fondo o dejar huecos en el envase; el porcentaje seleccionado es área de imagen, no precisión ni porcentaje de producto detectado.

In [ ]:
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
limite_inferior = np.array([30, 50, 35], dtype=np.uint8)
limite_superior = np.array([100, 255, 255], dtype=np.uint8)
mascara_verde = cv2.inRange(hsv, limite_inferior, limite_superior)
segmentada = cv2.bitwise_and(rgb, rgb, mask=mascara_verde)
porcentaje_area = 100 * np.count_nonzero(mascara_verde) / mascara_verde.size
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, titulo, cmap in zip(ejes, [rgb, mascara_verde, segmentada], ['Original', 'Máscara verde', f'Aplicada ({porcentaje_area:.1f}% del área)'], [None, 'gray', None]):
    ax.imshow(img, cmap=cmap); ax.set_title(titulo); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Límites de este análisis
- El conjunto tiene 50 fotos individuales en 11 clases provisionales y 71 escenas multiclase de supermercado.
- Las condiciones de captura no están registradas; las vistas no son un experimento controlado de iluminación.
- No hay cajas ni máscaras manuales propias para calcular exactitud.
- La máscara HSV es una línea base visual y debe compararse con una anotación de referencia.
- El próximo avance es anotar cajas/máscaras en las escenas, confirmar las clases provisionales y solicitar/descargar MVTec D2S.